# Exploratory Analysis - Titanic Survivors

## Validating Hypothesis #1

### **Analyzing Categorical Variables**

<img src="https://media1.giphy.com/media/JcFUHp7b9mnj5a01AN/giphy.gif" width="500"/>


### Business Goal

> Identify the factors that led some Titanic passengers to survive and others not.
>
> `Reason`: make future cruises safer.

### Notebook Goal

> 1. Evaluate Categorical Variables (Counts)
> 1. Visualize Categorical Data (Pie and Bar Charts)
> 1. Validate the Hypothesis (Hunch)
> 1. Validate the Hypothesis Statistically (p-value)


### 🗺️ Mapped Hypotheses

> 🔵: Hypotheses covered in this notebook

Passengers more likely to survive:
1. From higher classes (`Pclass` 1 > 2 > 3) 🔵
2. With more expensive tickets (`Fare`)
3. With distinguished titles (`Name`)
4. Aged 20-35 (`Age`)
5. Female (`Sex`)
6. With small families (`Parch` & `SibSp`)

Bonus:
7. Larger families are from lower classes
8. Higher classes have older passengers

![Hipóteses](https://c.tenor.com/Nq-PcKRJUFsAAAAC/bart-simpson-reading.gif)



### Install external dependencies

In [ ]:
!pip install --force-reinstall git+https://github.com/murilobellatini/eda

### Import Libs

In [ ]:
# custom libs
from src.paths import RAW_DATA_PATH_TITANIC, PROCESSED_DATA_PATH_TITANIC
from src.io import load_df_from_url, mount_gdrive

# visualization libs
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# statistics libs
from scipy.stats import ttest_ind
import numpy as np

sns.set()

### Load DataFrame

In [ ]:
# mount Google Drive and load `raw/train.csv`
mount_gdrive()
df = pd.read_csv(f'{RAW_DATA_PATH_TITANIC}/train.csv', index_col=0)
df

### 💓 Distribution of Survivors (Overall)

> How many passengers survived? And compared to the total?

In [ ]:
# count the number of survivors
survived = df.Survived.value_counts() # counts for categorical variables
survived

In [ ]:
# show the proportion of survivors
survived_norm = df.Survived.value_counts(normalize=True) * 100
survived_norm

In [ ]:
# join into a single table
pd.concat([
           survived,
           survived_norm.rename('%')
], axis=1)

# 💡 Technical Tip!!

> Always look at the `output` of your code.
>
> Every transformation you make should be validated.
>
> Especially when you are starting out.
>
> `Reason`
>
> * You will understand what is going on....
> * And learn faster! 🤓

<img src="https://c.tenor.com/9fJnzNoBiWUAAAAd/can-you-see-it-now-close.gif" width=500/>

> *I will read every line of my `output`...*

In [ ]:
# evaluate other statistics of survivors
df.Survived.describe()

In [ ]:
# rename the Series index
survived_norm = survived_norm.rename({0: 'Não Sobreviveu', 1: 'Sobreviveu'})
survived_norm

In [ ]:
# visualize the distribution of survivors
_ = (survived_norm
        .plot(
            kind='bar', table=True, legend=True, 
            title='Distribuição de Sobreviventes Macro',
            ylabel='% Todos Passageiros',
            fontsize=12, xticks=(),
            figsize=(7,7)
      )
    )

# 💡 Technical Tip!!

> Your charts must **always** explain themselves!
>
> That is, whoever reads them must be able to understand everything alone
>
> `Reason`
>
> 1. You will spend less time explaining
> 2. You will understand your data better in the process
> 3. Your chart can be read WITHOUT you
>
> `How to get there?`
>
> **Always** include:
>
> 1. Title
> 2. Legend
> 3. Axis names
> 4. Clear scale

<img src="https://i.pinimg.com/originals/c0/7f/3f/c07f3fe2f907344587c79af10d1c375f.gif" width=500/>

> *Did my job, bye...*

In [ ]:
# visualize as a pie chart
_ = survived_norm.plot(
    kind='pie', table=True, legend=True,
    title='Distribuição de Sobreviventes Macro',
    figsize=(7,7)
    )

# 💡 Technical Tip!!

> The pie chart (`pie chart`) is frowned upon in academia and in the data community.
> 
> `Reason`: Hard to read with many classes.
>
> `Solution`: Bar chart.

<img src="https://i.imgur.com/riXZoZS.gif" width=500/>

# Validating Hypothesis #1
> `Hypothesis`: Passengers from higher classes are more likely to survive (`Pclass` 1 > 2 > 3)

In [ ]:
#  show the mean of Survived
df.Survived.mean()

# 💡 Technical Tip!!

> The mean of a boolean column (0 or 1) is the percentage of positive cases.

In [ ]:
# show the mean of `Survived` by `Pclass`
survived_pclass = df.groupby('Pclass').Survived.mean().rename('Survived_perc')
survived_pclass 

In [ ]:
# visualize the mean of `Survived` by `Pclass`
_ = (100*survived_pclass).plot(
    kind='bar', table=True, legend=True,
    title='Distribuição de Sobreviventes por Classe',
    ylabel='% Sobreviventes', xlabel='\n\nClasse do Passageiro',
    fontsize=14, xticks=(),
    figsize=(7,7)
    )

# 👌 Insight / Conclusion

Hypothesis #1 validated. ✔️

> The higher the passenger's class, the higher the chance of survival.

<img src="https://media2.giphy.com/media/BfrOzZpsjfuiQ/200.gif" width=500>

> *I am very rich.*

# Hold on! Is this really true?

Even though the difference in survivors is very visual, we may be wrong.

How could we prove that there really is a difference between the samples?

After all, the difference may be within the error...

<img src="https://y.yarn.co/400c0671-6c05-4010-bd16-ed990f3bdb11_text.gif" width=500>

> Not so fast, Sally!


## **Bonus: How to prove it statistically?**

Let's talk about the `p-value`, or `Pearson value`.

It is widely used in academia.

It is used to compare distributions (data samples).

If it returns a very small value, we have shown that two samples are different.

Otherwise, they come from the same distribution (or data source).

***Very small = 0.05 or 5% (convention)***

<img src="https://c.tenor.com/cZ_8ynUgFzAAAAAM/frink-professor-frink.gif" width=500>

> This is the Null Hypothesis test, for the nerdiest. 🤓



In [ ]:
# create a method to compare distributions with `p-value`
def compare2groups(arr1, arr2, sample_size=150, alpha=0.05):
  """
  Compara se dois grupos de dados tem origem da mesma distribuição.
  Método: Pearson-value (p-value)
  """
  arr1_sampled = np.random.choice(arr1, sample_size)
  arr2_sampled = np.random.choice(arr2, sample_size)
  stat, p = ttest_ind(arr1_sampled, arr2_sampled)
  
  print(f't-statistic: {stat}\np-value: {p}')

  if p > alpha:
    print('Same Distribution (null hypothesis not rejected)')
  else:
    print('Different Distribution (null hypothesis rejected)')

In [ ]:
# compare two samples of 1st Class survivors
compare2groups(
    df[df.Pclass==1].Survived,
    df[df.Pclass==1].Survived)

# 💡 Technical Tip!!

> 🧠 A Sanity Check is always welcome.
>
> `Reason`
>
> > Make sure functions are implemented correctly.
>
> `How?`
> >
> > With a validated method, two samples from the same distribution MUST return a high `p-value` (above 5%).
> >
> > That is, the test confirms the samples come from the same distribution.
> >
> > This was expected!
> >
> > The implementation must be correct.

<img src="https://media1.giphy.com/media/z23hGvopHu7w4/giphy.gif" width=500>

> *Phew...*

In [ ]:
# compare samples of 1st and 2nd Class survivors
compare2groups(
    df[df.Pclass==1].Survived,
    df[df.Pclass==2].Survived)

In [ ]:
# compare samples of 2nd and 3rd Class survivors
compare2groups(
    df[df.Pclass==2].Survived,
    df[df.Pclass==3].Survived)

# 👌 Insight / Conclusion

Hypothesis #1 confirmed with STATISTICAL validity. ✔️

<img src="https://media0.giphy.com/media/l4q8cJzGdR9J8w3hS/giphy.gif" width=500>


## Summary

In this lesson we learned:

1. Count categorical variables
2. Visualize data as bars (`bar chart`)
3. Visualize data as a pie (`pie chart`)
4. How to validate our 1st hypothesis in practice
5. Bonus: How to validate with statistical rigor (p-value)

<img src="https://media1.giphy.com/media/3ov9k01Y5IKizNmC7S/200.gif" width="500"/>


# Done!

That's all for this lesson...

<img src="https://c.tenor.com/IAUvvrUY7zQAAAAM/done-spongebob.gif" width="500"/>

> *Done*